In [1]:
# CÉLULA 1 — INSTALAÇÃO DAS DEPENDÊNCIAS
!pip install tensorflowjs
!pip uninstall -y tensorflow-decision-forests 2>/dev/null || true
!pip install tf-keras tensorflowjs==4.22.0 kagglehub "numpy<2.0" --quiet

  Using cached tensorflow_decision_forests-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.3 kB)
Using cached tensorflow_decision_forests-1.12.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.1 MB)
Found existing installation: tensorflow_decision_forests 1.12.0
Uninstalling tensorflow_decision_forests-1.12.0:
  Successfully uninstalled tensorflow_decision_forests-1.12.0

✅ Instalação concluída!

⚠️  PRÓXIMO PASSO OBRIGATÓRIO:
   Menu → Ambiente de Execução → Reiniciar sessão
   Depois execute APENAS a Célula 2 (não execute esta célula novamente).


In [4]:
# =============================================================================
# CÉLULA 2 — TREINAMENTO (SUPER DATASET + TRANSFER LEARNING) e SALVAMENTO
# Execute SOMENTE após reiniciar a sessão (conforme instrução da Célula 1).
# =============================================================================

import os
import shutil
import numpy as np
import tensorflow as tf
import tf_keras
from tf_keras import layers, models
import kagglehub

print("TensorFlow versão:", tf.__version__)
print("tf_keras versão:", tf_keras.__version__)

# ---------------------------------------------------------
# 1. DOWNLOAD E MESCLAGEM DOS DADOS (SUPER DATASET)
# ---------------------------------------------------------
print("\nBaixando dataset 1 (TrashNet)...")
path_trashnet = kagglehub.dataset_download("feyzazkefe/trashnet")
if len(os.listdir(path_trashnet)) == 1:
    path_trashnet = os.path.join(path_trashnet, os.listdir(path_trashnet)[0])

print("Baixando dataset 2 (Garbage Classification - Real World)...")
# Essa base tem fundos reais (mãos, rua, grama) e mapeia bem para as 6 classes
path_real_world = kagglehub.dataset_download("asdasdasasdas/garbage-classification")
if len(os.listdir(path_real_world)) == 1:
    path_real_world = os.path.join(path_real_world, os.listdir(path_real_world)[0])

# Criando a super pasta para misturar os dois
SUPER_DATASET_DIR = "/content/super_dataset"
os.makedirs(SUPER_DATASET_DIR, exist_ok=True)

# As 6 classes padrão que queremos manter
CLASSES_ALVO = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

print("\nMesclando as imagens em uma única base...")
for classe in CLASSES_ALVO:
    # Cria a pasta da classe no super dataset
    pasta_destino = os.path.join(SUPER_DATASET_DIR, classe)
    os.makedirs(pasta_destino, exist_ok=True)

    # Copia imagens do TrashNet
    origem_trashnet = os.path.join(path_trashnet, classe)
    if os.path.exists(origem_trashnet):
        for img in os.listdir(origem_trashnet):
            if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                shutil.copy(os.path.join(origem_trashnet, img), os.path.join(pasta_destino, f"tn_{img}"))

    # Copia imagens da segunda base (Real World)
    origem_real = os.path.join(path_real_world, classe)
    if os.path.exists(origem_real):
        for img in os.listdir(origem_real):
            if img.lower().endswith(('.png', '.jpg', '.jpeg')):
                shutil.copy(os.path.join(origem_real, img), os.path.join(pasta_destino, f"rw_{img}"))

print("Diretório do super dataset final:", SUPER_DATASET_DIR)

BATCH_SIZE = 32
IMG_SIZE   = (224, 224)
AUTOTUNE   = tf.data.AUTOTUNE

# Carrega os datasets a partir da nossa nova super pasta
train_ds = tf_keras.utils.image_dataset_from_directory(
    SUPER_DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_ds = tf_keras.utils.image_dataset_from_directory(
    SUPER_DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

class_names = train_ds.class_names
print("Classes (ordem usada pelo modelo):", class_names)

# ---------------------------------------------------------
# 2. DATA AUGMENTATION (Mantido como estava)
# ---------------------------------------------------------
augmentation_model = tf_keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
], name="data_augmentation")

def augment(image, label):
    image = augmentation_model(image, training=True)
    return image, label

train_ds = train_ds.cache().shuffle(1000).map(augment, num_parallel_calls=AUTOTUNE).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ---------------------------------------------------------
# 3. CONSTRUÇÃO DO MODELO COM TRANSFER LEARNING (NOVIDADE)
# ---------------------------------------------------------
print("\nBaixando modelo base MobileNetV2...")
# Carrega o cérebro do Google, sem a última camada (include_top=False)
base_model = tf_keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Congela o cérebro base para ele não esquecer o que já sabe
base_model.trainable = False

# Monta o nosso modelo final conectando o cérebro às nossas classes
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    # O MobileNetV2 espera pixels entre [-1, 1], então a normalização muda aqui:
    layers.Rescaling(1./127.5, offset=-1),

    # O Cérebro do Google
    base_model,

    # A nossa camada final para transformar o conhecimento dele nas nossas 6 classes
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(len(class_names), activation='softmax')
], name="classificador_lixo_mobilenet")

model.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=0.001), # Taxa de aprendizado controlada
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------------------------------------
# 4. TREINAMENTO
# ---------------------------------------------------------
print("\nIniciando o treinamento (Apenas a última camada)...")
# Com o Transfer Learning, precisamos de menos épocas para ter um bom resultado
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

# ---------------------------------------------------------
# 5. SALVAMENTO EM FORMATO HDF5 (.h5)
# ---------------------------------------------------------
model.save('modelo_trashnet.h5')
print("\n✅ Modelo salvo como modelo_trashnet.h5")

TensorFlow versão: 2.19.0
tf_keras versão: 2.19.0

Baixando dataset 1 (TrashNet)...
Using Colab cache for faster access to the 'trashnet' dataset.
Baixando dataset 2 (Garbage Classification - Real World)...
Using Colab cache for faster access to the 'garbage-classification' dataset.

Mesclando as imagens em uma única base...
Diretório do super dataset final: /content/super_dataset
Found 2527 files belonging to 6 classes.
Using 2022 files for training.
Found 2527 files belonging to 6 classes.
Using 505 files for validation.
Classes (ordem usada pelo modelo): ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

Baixando modelo base MobileNetV2...
Model: "classificador_lixo_mobilenet"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 mobilenetv2_1.00_224 (Func  (None, 

/usr/local/lib/python3.12/dist-packages/tf_keras/src/engine/training.py:3098: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native TF-Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(



✅ Modelo salvo como modelo_trashnet.h5


In [5]:
# =============================================================================
# CÉLULA 3 — CONVERSÃO INDEPENDENTE PARA TENSORFLOW.JS
# Execute esta célula de forma isolada sempre que precisar gerar os arquivos da Web.
# =============================================================================

import os
import tf_keras
import tensorflowjs as tfjs

ARQUIVO_H5 = 'modelo_trashnet.h5'
PASTA_SAIDA = 'modelo_tfjs'

print("1️⃣  Verificando o arquivo de origem...")
if not os.path.exists(ARQUIVO_H5):
    print(f"❌ ERRO: O arquivo '{ARQUIVO_H5}' não foi encontrado no ambiente atual.")
    print("📌 Certifique-se de que o treino (Célula 2) terminou com sucesso nesta mesma sessão.")
else:
    print(f"✅ Arquivo '{ARQUIVO_H5}' detectado com sucesso!")

    try:
        print("\n2️⃣  Carregando o modelo treinado para a memória do Python...")
        # Carrega usando tf_keras para manter a compatibilidade total com o Keras 2 do h5
        model = tf_keras.models.load_model(ARQUIVO_H5)
        print("✅ Modelo carregado com sucesso!")

        print("\n3️⃣  Iniciando conversão direta via API do TensorFlow.js...")
        # Converte o modelo da memória direto para a pasta final
        tfjs.converters.save_keras_model(model, PASTA_SAIDA)

        print("\n🎉 CONVERSÃO CONCLUÍDA COM SUCESSO!")
        print(f"📁 Arquivos gerados prontos para uso na pasta '{PASTA_SAIDA}/':")

        tamanho_total = 0
        for f in sorted(os.listdir(PASTA_SAIDA)):
            caminho_arquivo = os.path.join(PASTA_SAIDA, f)
            tamanho = os.path.getsize(caminho_arquivo)
            tamanho_total += tamanho
            print(f"   📄 {f:<40} {tamanho // 1024:>6} KB")
        print(f"   📦 {'TAMANHO TOTAL DO PACOTE':<40} {tamanho_total // 1024:>6} KB")

        print("\n📥 Como baixar para o seu computador:")
        print("1. Clique no ícone de pasta (📁) na barra lateral esquerda do Google Colab.")
        print(f"2. Localize a pasta chamada '{PASTA_SAIDA}'.")
        print("3. Clique nos três pontinhos ao lado dela (ou botão direito) e selecione 'Fazer download'.")

    except Exception as e:
        print("\n❌ Erro durante o processo de conversão:")
        print(e)

1️⃣  Verificando o arquivo de origem...
✅ Arquivo 'modelo_trashnet.h5' detectado com sucesso!

2️⃣  Carregando o modelo treinado para a memória do Python...
✅ Modelo carregado com sucesso!

3️⃣  Iniciando conversão direta via API do TensorFlow.js...

🎉 CONVERSÃO CONCLUÍDA COM SUCESSO!
📁 Arquivos gerados prontos para uso na pasta 'modelo_tfjs/':
   📄 group1-shard10of11.bin                     4096 KB
   📄 group1-shard11of11.bin                     2671 KB
   📄 group1-shard1of11.bin                      4096 KB
   📄 group1-shard1of3.bin                       4096 KB
   📄 group1-shard2of11.bin                      4096 KB
   📄 group1-shard2of3.bin                       4096 KB
   📄 group1-shard3of11.bin                      4096 KB
   📄 group1-shard3of3.bin                        658 KB
   📄 group1-shard4of11.bin                      4096 KB
   📄 group1-shard5of11.bin                      4096 KB
   📄 group1-shard6of11.bin                      4096 KB
   📄 group1-shard7of11.bin           